In [1]:
# 01 — CLEANING SETUP & INPUT VALIDATION
# OFFLINE ONLY
# ============================================================

from pathlib import Path
import pandas as pd
import re
import html
import unicodedata

ROOT = Path(r"C:\Penelitian Sentimen Bencana Nasional")

INPUT_FILE = (
    ROOT / "01_Raw Data"
    / "outputs" / "reports" / "batch7"
    / "manual_reconciliation"
    / "Batch7_Eligible_Tweets.csv"
)

OUTPUT_DIR = ROOT / "02_Cleaning" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert INPUT_FILE.exists(), (
    f"Input tidak ditemukan: {INPUT_FILE}"
)

df = pd.read_csv(
    INPUT_FILE,
    dtype={"id": "string"},
    keep_default_na=False,
    encoding="utf-8-sig"
)

assert len(df) == 381, (
    f"Expected 381 tweet, ditemukan {len(df)}"
)

assert df["id"].is_unique, "Duplicate tweet ID."
assert df["text"].astype(str).str.strip().ne("").all()
assert df["final_inclusion_decision"].eq("INCLUDE").all()

print("CLEANING INPUT VALIDATION: PASS")
print("Eligible tweets:", len(df))
print("\nDistribution by case:")
display(df["case"].value_counts())


CLEANING INPUT VALIDATION: PASS
Eligible tweets: 381

Distribution by case:


case
sumatra_2025    149
palu_2018       118
lombok_2018     114
Name: count, dtype: int64

In [2]:
# 02 — INITIAL TEXT QUALITY AUDIT
# OFFLINE ONLY | NO TEXT MODIFICATION
# ============================================================

import pandas as pd
import re
import html
import unicodedata

assert len(df) == 381, "Input harus 381 tweet."
assert df["id"].is_unique, "Ada ID duplikat."

audit = df.copy()

# Simpan referensi teks asli tanpa perubahan
audit["text_original"] = audit["text"].astype("string")

# Pemeriksaan dasar
audit["text_length"] = audit["text_original"].str.len()
audit["is_empty"] = (
    audit["text_original"].str.strip().eq("")
)

audit["has_url"] = audit["text_original"].str.contains(
    r"https?://|www\.",
    regex=True,
    case=False,
    na=False
)

audit["has_mention"] = audit["text_original"].str.contains(
    r"(?<!\w)@\w+",
    regex=True,
    na=False
)

audit["has_hashtag"] = audit["text_original"].str.contains(
    r"(?<!\w)#\w+",
    regex=True,
    na=False
)

audit["has_html_entity"] = audit["text_original"].str.contains(
    r"&(?:amp|lt|gt|quot|apos|#\d+|#x[0-9a-f]+);",
    regex=True,
    case=False,
    na=False
)

# Duplikasi exact berdasarkan teks
audit["duplicate_exact_text"] = (
    audit.duplicated(
        subset=["text_original"],
        keep=False
    )
)

# Duplikasi setelah normalisasi ringan:
# Unicode NFKC, HTML unescape, lowercase, whitespace.
# Belum menghapus token ataupun mengubah teks original.
def comparison_key(text):
    text = html.unescape(str(text))
    text = unicodedata.normalize("NFKC", text)
    text = text.casefold()
    text = re.sub(r"\s+", " ", text).strip()
    return text

audit["text_comparison_key"] = (
    audit["text_original"].map(comparison_key)
)

audit["duplicate_normalized_text"] = (
    audit.duplicated(
        subset=["text_comparison_key"],
        keep=False
    )
)

summary = pd.DataFrame([
    ["Total tweets", len(audit)],
    ["Unique tweet IDs", audit["id"].nunique()],
    ["Empty text", int(audit["is_empty"].sum())],
    ["Exact duplicate rows", int(audit["duplicate_exact_text"].sum())],
    ["Normalized duplicate rows", int(audit["duplicate_normalized_text"].sum())],
    ["Contains URL", int(audit["has_url"].sum())],
    ["Contains mention", int(audit["has_mention"].sum())],
    ["Contains hashtag", int(audit["has_hashtag"].sum())],
    ["Contains HTML entity", int(audit["has_html_entity"].sum())],
    ["Minimum text length", int(audit["text_length"].min())],
    ["Maximum text length", int(audit["text_length"].max())],
], columns=["Metric", "Value"])

print("=" * 60)
print("02 — INITIAL TEXT QUALITY AUDIT")
print("=" * 60)
display(summary)

print("\nExact duplicate samples:")
display(
    audit.loc[
        audit["duplicate_exact_text"],
        ["case", "id", "text_original"]
    ].head(15)
)

print("\nNormalized duplicate samples:")
display(
    audit.loc[
        audit["duplicate_normalized_text"],
        ["case", "id", "text_original"]
    ].head(15)
)

print("\nText length statistics:")
display(
    audit.groupby("case")["text_length"]
    .describe()
)

assert not audit["is_empty"].any(), (
    "Ada teks kosong yang perlu diperiksa."
)

print("\nINITIAL TEXT QUALITY AUDIT: PASS")
print("Original text: UNCHANGED")
print("No API call. No data deletion.")


02 — INITIAL TEXT QUALITY AUDIT


,Metric,Value
0,Total tweets,381
1,Unique tweet IDs,381
2,Empty text,0
3,Exact duplicate rows,7
4,Normalized duplicate rows,7
5,Contains URL,232
6,Contains mention,135
7,Contains hashtag,103
8,Contains HTML entity,24
9,Minimum text length,61



Exact duplicate samples:


,case,id,text_original
42,lombok_2018,1028971930978840577,Jika gempa Lombok ditetapkan sebagai bencana n...
43,lombok_2018,1028971930945298433,Jika gempa Lombok ditetapkan sebagai bencana n...
44,lombok_2018,1028968156587540480,Apalagi hingga kini pemerintah pusat belum men...
45,lombok_2018,1028968156130357248,Apalagi hingga kini pemerintah pusat belum men...
74,sumatra_2025,1996195214365757833,Bencana banjir serta longsor melanda sebagian ...
75,sumatra_2025,1996180113290088659,Bencana banjir serta longsor melanda sebagian ...
81,sumatra_2025,1996149775524376777,Bencana banjir serta longsor melanda sebagian ...



Normalized duplicate samples:


,case,id,text_original
42,lombok_2018,1028971930978840577,Jika gempa Lombok ditetapkan sebagai bencana n...
43,lombok_2018,1028971930945298433,Jika gempa Lombok ditetapkan sebagai bencana n...
44,lombok_2018,1028968156587540480,Apalagi hingga kini pemerintah pusat belum men...
45,lombok_2018,1028968156130357248,Apalagi hingga kini pemerintah pusat belum men...
74,sumatra_2025,1996195214365757833,Bencana banjir serta longsor melanda sebagian ...
75,sumatra_2025,1996180113290088659,Bencana banjir serta longsor melanda sebagian ...
81,sumatra_2025,1996149775524376777,Bencana banjir serta longsor melanda sebagian ...



Text length statistics:


,count,mean,std,min,25%,50%,75%,max
case,,,,,,,,
lombok_2018,114.0,164.508772,72.839409,61.0,107.25,142.5,231.5,301.0
palu_2018,118.0,176.567797,76.950587,73.0,98.0,164.0,252.5,362.0
sumatra_2025,149.0,474.550336,752.640509,73.0,197.0,282.0,399.0,6033.0



INITIAL TEXT QUALITY AUDIT: PASS
Original text: UNCHANGED
No API call. No data deletion.


In [3]:
# 03 — SAFE TEXT NORMALIZATION + LONG TEXT AUDIT
# OFFLINE ONLY | NO DATA DELETION
# ============================================================

import re
import html
import unicodedata
import pandas as pd

assert len(df) == 381
assert df["id"].is_unique
assert "text_original" in audit.columns

clean_df = audit.copy()

# Simpan teks asli dan metadata.
clean_df["text_original"] = (
    clean_df["text_original"].astype("string")
)

def normalize_basic_text(value):
    if pd.isna(value):
        return ""

    text = str(value)

    # Decode HTML entities berulang secara terbatas.
    # Misalnya &amp;amp; -> &
    for _ in range(3):
        decoded = html.unescape(text)
        if decoded == text:
            break
        text = decoded

    # Normalisasi Unicode.
    text = unicodedata.normalize("NFC", text)

    # Hilangkan karakter kontrol, kecuali whitespace
    # yang akan dinormalisasi di bawah.
    text = "".join(
        " " if (
            unicodedata.category(char) == "Cc"
        ) else char
        for char in text
    )

    # Hapus zero-width spacing yang sering tidak terlihat.
    text = re.sub(
        "[\u200b\u200c\u200d\ufeff]",
        "",
        text
    )

    # Rapikan seluruh whitespace.
    text = re.sub(r"\s+", " ", text).strip()

    return text

# Kolom hasil normalisasi, tanpa overwrite teks asli.
clean_df["text_normalized"] = (
    clean_df["text_original"]
    .map(normalize_basic_text)
    .astype("string")
)

# Audit perubahan.
clean_df["original_length"] = (
    clean_df["text_original"].str.len()
)
clean_df["normalized_length"] = (
    clean_df["text_normalized"].str.len()
)

clean_df["normalization_changed"] = (
    clean_df["text_original"]
    != clean_df["text_normalized"]
)

clean_df["length_difference"] = (
    clean_df["original_length"]
    - clean_df["normalized_length"]
)

# Flag teks panjang untuk pemeriksaan khusus.
LONG_TEXT_THRESHOLD = 1000

clean_df["is_long_text"] = (
    clean_df["original_length"]
    > LONG_TEXT_THRESHOLD
)

# Tidak boleh ada tweet hilang atau kosong.
assert len(clean_df) == len(df) == 381
assert clean_df["id"].is_unique
assert clean_df["text_normalized"].str.strip().ne("").all()
assert set(clean_df["id"]) == set(df["id"])

# Pastikan teks asli tidak berubah.
original_by_id = (
    audit.set_index("id")["text_original"]
    .astype("string")
)

current_by_id = (
    clean_df.set_index("id")["text_original"]
    .astype("string")
)

assert current_by_id.sort_index().equals(
    original_by_id.sort_index()
), "Teks asli berubah."

print("=" * 62)
print("03 — SAFE TEXT NORMALIZATION AUDIT")
print("=" * 62)
print("Total tweets       :", len(clean_df))
print(
    "Changed by normalization:",
    int(clean_df["normalization_changed"].sum())
)
print(
    "Unchanged              :",
    int((~clean_df["normalization_changed"]).sum())
)
print(
    "Long texts (>1000)     :",
    int(clean_df["is_long_text"].sum())
)
print(
    "Empty normalized texts :",
    int(clean_df["text_normalized"].str.strip().eq("").sum())
)

print("\nDistribution of long texts by case:")
display(
    clean_df.groupby("case")["is_long_text"]
    .agg(["sum", "count"])
)

print("\nExamples of changed text:")
display(
    clean_df.loc[
        clean_df["normalization_changed"],
        [
            "case",
            "id",
            "text_original",
            "text_normalized"
        ]
    ].head(10)
)

print("\nLong-text audit:")
display(
    clean_df.loc[
        clean_df["is_long_text"],
        [
            "case",
            "id",
            "original_length",
            "normalized_length",
            "text_original"
        ]
    ].sort_values(
        "original_length",
        ascending=False
    ).head(15)
)

print("\nCELL 03 NORMALIZATION: PASS")
print("Original data: PRESERVED")
print("Rows removed: 0")
print("No API calls.")


03 — SAFE TEXT NORMALIZATION AUDIT
Total tweets       : 381
Changed by normalization: 221
Unchanged              : 160
Long texts (>1000)     : 12
Empty normalized texts : 0

Distribution of long texts by case:


,sum,count
case,,
lombok_2018,0,114
palu_2018,0,118
sumatra_2025,12,149



Examples of changed text:


,case,id,text_original,text_normalized
7,lombok_2018,1026402242776117249,Wakil Ketua DPR RI Fahri Hamzah meminta pemeri...,Wakil Ketua DPR RI Fahri Hamzah meminta pemeri...
8,lombok_2018,1026398913371074562,Bangda @Fahrihamzah Minta Status Gempa di Lomb...,Bangda @Fahrihamzah Minta Status Gempa di Lomb...
10,lombok_2018,1026372308284264448,Fahri Minta Status Gempa Lombok Jadi Bencana N...,Fahri Minta Status Gempa Lombok Jadi Bencana N...
11,palu_2018,1046068799915642880,@Jogja_Uncover ASTAGFIRULLOHALAZIM😭😭\nAmpuni k...,@Jogja_Uncover ASTAGFIRULLOHALAZIM😭😭 Ampuni ka...
13,palu_2018,1046044447295885312,@dapitdong Kami mendesak Palu - Donggala seger...,@dapitdong Kami mendesak Palu - Donggala seger...
14,palu_2018,1046042304354689027,@liputan6dotcom Kami mendesak Palu - Donggala ...,@liputan6dotcom Kami mendesak Palu - Donggala ...
15,palu_2018,1046031572699205633,Setelah Bencana Lombok ditolak menjadi Bencana...,Setelah Bencana Lombok ditolak menjadi Bencana...
18,palu_2018,1046016344519397376,Kami mendesak Palu - Donggala segera ditetapka...,Kami mendesak Palu - Donggala segera ditetapka...
20,palu_2018,1045955687405113344,Setuju... untuk mempercepat proses pertolongan...,Setuju... untuk mempercepat proses pertolongan...
21,palu_2018,1045880601759252482,Sudah selayaknya bencana gempa yg disertai tsu...,Sudah selayaknya bencana gempa yg disertai tsu...



Long-text audit:


,case,id,original_length,normalized_length,text_original
208,sumatra_2025,1995658121684152682,6033,5978,"Saat Aceh dan Sumatera ""Tenggelam"", Percakapan..."
147,sumatra_2025,1996389246203212130,5449,5418,Saya paham jika ada kepala daerah gelisah deng...
129,sumatra_2025,1996110111954329621,3475,3451,Ketika seorang ulama besar menyebut di depan r...
141,sumatra_2025,1996433134452375771,2385,2372,811 NYAWA DAN LUMPUR YANG TAK KUNJUNG KERING\n...
27,sumatra_2025,1994716450863579467,2132,2127,Bencana nasional yang terjadi di Sumatera bebe...
179,sumatra_2025,1995764391585919009,1354,1350,"Banjir dan tanah longsor yang melanda Aceh, Su..."
206,sumatra_2025,1995680229629776022,1345,1332,KLARIFIKASI HOAX - MISLEADING CONTENT \n[SALAH...
193,sumatra_2025,1995824414844485717,1253,1248,"Anggota Komisi V DPR RI Fraksi PKS, Abdul Hadi..."
112,sumatra_2025,1996059159654248664,1177,1172,"Bencana banjir bandang dan longsor di Aceh, Su..."
144,sumatra_2025,1996407464045211902,1127,1121,Presiden @prabowo diframing sedemikian oleh pe...



CELL 03 NORMALIZATION: PASS
Original data: PRESERVED
Rows removed: 0
No API calls.


In [4]:
# 04 — SOCIAL MEDIA TEXT CLEANING
# URL, MENTION, RT PREFIX, HASHTAG
# OFFLINE ONLY | ORIGINAL PRESERVED
# ============================================================

import re
import pandas as pd

assert len(clean_df) == 381
assert clean_df["id"].is_unique
assert "text_original" in clean_df.columns
assert "text_normalized" in clean_df.columns

social_df = clean_df.copy()

# ------------------------------------------------------------
# A. REGEX PATTERNS
# ------------------------------------------------------------

URL_PATTERN = re.compile(
    r"(?i)\b(?:https?://|www\.)[^\s<>]+"
)

MENTION_PATTERN = re.compile(
    r"(?<!\w)@[A-Za-z0-9_]{1,15}\b"
)

RT_PREFIX_PATTERN = re.compile(
    r"(?i)^\s*RT\s+"
)

HASHTAG_PATTERN = re.compile(
    r"(?<!\w)#([^\W_][\w]*)",
    flags=re.UNICODE
)

# ------------------------------------------------------------
# B. CLEANING FUNCTION
# ------------------------------------------------------------

def clean_social_text(value):
    if pd.isna(value):
        return ""

    text = str(value)

    # Hapus prefix retweet di awal teks.
    text = RT_PREFIX_PATTERN.sub("", text)

    # Hapus URL.
    text = URL_PATTERN.sub(" ", text)

    # Hapus mention, bukan konten kalimat.
    text = MENTION_PATTERN.sub(" ", text)

    # Pertahankan isi hashtag.
    text = HASHTAG_PATTERN.sub(r"\1", text)

    # Bersihkan separator retweet yang tertinggal
    # hanya bila berada di awal teks.
    text = re.sub(r"^\s*:\s*", "", text)

    # Rapikan spasi.
    text = re.sub(r"\s+", " ", text).strip()

    return text

# ------------------------------------------------------------
# C. APPLY CLEANING
# ------------------------------------------------------------

social_df["text_social_clean"] = (
    social_df["text_normalized"]
    .map(clean_social_text)
    .astype("string")
)

# ------------------------------------------------------------
# D. AUDIT CHANGES
# ------------------------------------------------------------

social_df["social_changed"] = (
    social_df["text_normalized"]
    != social_df["text_social_clean"]
)

social_df["social_length_before"] = (
    social_df["text_normalized"].str.len()
)

social_df["social_length_after"] = (
    social_df["text_social_clean"].str.len()
)

social_df["social_chars_removed"] = (
    social_df["social_length_before"]
    - social_df["social_length_after"]
)

social_df["social_empty"] = (
    social_df["text_social_clean"]
    .str.strip()
    .eq("")
)

# Flag kata penting terkait fokus penelitian.
POLICY_TERMS = [
    "bencana nasional",
    "status bencana",
    "penetapan",
    "pemerintah",
    "menolak",
    "setuju",
    "tidak",
    "bukan",
    "belum",
]

def present_terms(text):
    lowered = str(text).casefold()
    return [
        term for term in POLICY_TERMS
        if term in lowered
    ]

social_df["policy_terms_before"] = (
    social_df["text_normalized"]
    .map(present_terms)
)

social_df["policy_terms_after"] = (
    social_df["text_social_clean"]
    .map(present_terms)
)

social_df["policy_terms_lost"] = [
    sorted(set(before) - set(after))
    for before, after in zip(
        social_df["policy_terms_before"],
        social_df["policy_terms_after"]
    )
]

social_df["needs_semantic_review"] = (
    social_df["policy_terms_lost"].map(bool)
    | (
        social_df["social_chars_removed"]
        > social_df["social_length_before"] * 0.5
    )
)

# ------------------------------------------------------------
# E. ASSERTIONS
# ------------------------------------------------------------

assert len(social_df) == len(clean_df) == 381
assert social_df["id"].is_unique
assert set(social_df["id"]) == set(clean_df["id"])

assert not social_df["social_empty"].any(), (
    "Ada tweet menjadi kosong setelah social cleaning."
)

assert social_df["text_original"].equals(
    clean_df["text_original"]
), "Original text berubah."

assert social_df["text_normalized"].equals(
    clean_df["text_normalized"]
), "Normalized text berubah."

# ------------------------------------------------------------
# F. REPORT
# ------------------------------------------------------------

print("=" * 62)
print("04 — SOCIAL MEDIA TEXT CLEANING AUDIT")
print("=" * 62)

print("Total tweets              :", len(social_df))
print(
    "Changed text              :",
    int(social_df["social_changed"].sum())
)
print(
    "Unchanged text            :",
    int((~social_df["social_changed"]).sum())
)
print(
    "Empty after cleaning      :",
    int(social_df["social_empty"].sum())
)
print(
    "Need semantic review      :",
    int(social_df["needs_semantic_review"].sum())
)
print(
    "Lost policy keyword rows  :",
    int(
        social_df["policy_terms_lost"]
        .map(bool)
        .sum()
    )
)

print("\nExamples of changed text:")
display(
    social_df.loc[
        social_df["social_changed"],
        [
            "case",
            "id",
            "text_normalized",
            "text_social_clean",
        ]
    ].head(12)
)

print("\nFlagged for semantic review:")
display(
    social_df.loc[
        social_df["needs_semantic_review"],
        [
            "case",
            "id",
            "social_chars_removed",
            "policy_terms_lost",
            "text_normalized",
            "text_social_clean",
        ]
    ].head(20)
)

print("\nSOCIAL MEDIA CLEANING: PASS")
print("Original preserved: YES")
print("Rows deleted: 0")
print("No API calls.")


04 — SOCIAL MEDIA TEXT CLEANING AUDIT
Total tweets              : 381
Changed text              : 337
Unchanged text            : 44
Empty after cleaning      : 0
Need semantic review      : 5
Lost policy keyword rows  : 0

Examples of changed text:


,case,id,text_normalized,text_social_clean
0,palu_2018,1047490881878511616,@joxzin_jogja Yang mendukung status Bencana Na...,Yang mendukung status Bencana Nasional Lombok ...
1,lombok_2018,1026504085548101632,@LawanPoLitikJKW Kalo pemerintah punya sense o...,"Kalo pemerintah punya sense of crisis, mustiny..."
3,lombok_2018,1026424653298855937,BNPB: Tak Perlu Status Bencana Nasional dan Ba...,BNPB: Tak Perlu Status Bencana Nasional dan Ba...
4,lombok_2018,1026421529775493121,Pemerintah Diminta Naikan Status Lombok Jadi B...,Pemerintah Diminta Naikan Status Lombok Jadi B...
5,lombok_2018,1026411069466533888,Fahri Hamzah Ingin Gempa Lombok Ditetapkan seb...,Fahri Hamzah Ingin Gempa Lombok Ditetapkan seb...
6,lombok_2018,1026403261195071489,DPR Minta Gempa Lombok jadi Bencana Nasional #...,DPR Minta Gempa Lombok jadi Bencana Nasional a...
7,lombok_2018,1026402242776117249,Wakil Ketua DPR RI Fahri Hamzah meminta pemeri...,Wakil Ketua DPR RI Fahri Hamzah meminta pemeri...
8,lombok_2018,1026398913371074562,Bangda @Fahrihamzah Minta Status Gempa di Lomb...,Bangda Minta Status Gempa di Lombok Jadi Benca...
9,lombok_2018,1026390946693271553,Gempa NTB Diminta jadi Bencana Nasional https:...,Gempa NTB Diminta jadi Bencana Nasional
10,lombok_2018,1026372308284264448,Fahri Minta Status Gempa Lombok Jadi Bencana N...,Fahri Minta Status Gempa Lombok Jadi Bencana N...



Flagged for semantic review:


,case,id,social_chars_removed,policy_terms_lost,text_normalized,text_social_clean
9,lombok_2018,1026390946693271553,48,[],Gempa NTB Diminta jadi Bencana Nasional https:...,Gempa NTB Diminta jadi Bencana Nasional
36,lombok_2018,1028974526841815041,56,[],Live! Gempa Lombok Bencana Nasional - @rakbir_...,Live! Gempa Lombok Bencana Nasional - bersama
87,palu_2018,1046820790853808128,144,[],Pemuda Muslimin Indonesia Harap Bencana Palu L...,Pemuda Muslimin Indonesia Harap Bencana Palu L...
282,lombok_2018,1028214950508916736,178,[],@Quvvatt @EV220622 @amrullah_sk @Reborn_Buroq ...,"Kalau lombok tidak dijadikan bencana nasional,..."
368,sumatra_2025,1995867615701676508,150,[],@Kimberley_PS08 @florieliciouss @are_inismynam...,"Naikan statusnya jadi bencana nasional, kita m..."



SOCIAL MEDIA CLEANING: PASS
Original preserved: YES
Rows deleted: 0
No API calls.


In [5]:
# 04A — SOCIAL CLEANING SEMANTIC REVIEW EXPORT
# OFFLINE ONLY | READ ONLY
# ============================================================

from pathlib import Path
import pandas as pd

assert len(social_df) == 381
assert social_df["id"].is_unique

AUDIT_DIR_04 = OUTPUT_DIR / "audits"
AUDIT_DIR_04.mkdir(parents=True, exist_ok=True)

REVIEW_PATH = (
    AUDIT_DIR_04 / "part4_semantic_review.xlsx"
)
FULL_AUDIT_PATH = (
    AUDIT_DIR_04 / "part4_full_audit.csv"
)

# Tidak menimpa file audit sebelumnya.
for path in [REVIEW_PATH, FULL_AUDIT_PATH]:
    if path.exists():
        raise FileExistsError(
            f"File sudah tersedia: {path}. "
            "Periksa atau arsipkan sebelum ekspor ulang."
        )

# A. Full audit seluruh tweet.
audit_cols = [
    "case",
    "id",
    "text_original",
    "text_normalized",
    "text_social_clean",
    "social_chars_removed",
    "needs_semantic_review",
]

full_audit = social_df[audit_cols].copy()
full_audit.to_csv(
    FULL_AUDIT_PATH,
    index=False,
    encoding="utf-8-sig"
)

# B. Audit khusus baris yang ditandai.
review = social_df.loc[
    social_df["needs_semantic_review"],
    audit_cols
].copy()

review["semantic_decision"] = ""
review["semantic_notes"] = ""
review["reviewer"] = ""
review["reviewed_at"] = ""

# C. Ringkasan.
summary = pd.DataFrame([
    ["Total eligible tweets", len(social_df)],
    ["Changed by social cleaning",
     int(social_df["social_changed"].sum())],
    ["Semantic review required", len(review)],
    ["Empty after cleaning",
     int(social_df["social_empty"].sum())],
], columns=["Metric", "Value"])

# Proteksi nilai teks agar tidak dieksekusi
# sebagai formula ketika dibuka di Excel.
def safe_excel(frame):
    result = frame.copy()
    for column in result.columns:
        result[column] = result[column].map(
            lambda value: "'" + value
            if isinstance(value, str)
            and value.startswith(("=", "+", "-", "@"))
            else value
        )
    return result

with pd.ExcelWriter(
    REVIEW_PATH,
    engine="openpyxl"
) as writer:
    safe_excel(summary).to_excel(
        writer, sheet_name="SUMMARY", index=False
    )

    safe_excel(review).to_excel(
        writer, sheet_name="SEMANTIC_REVIEW",
        index=False
    )

    ws = writer.sheets["SEMANTIC_REVIEW"]
    ws.freeze_panes = "C2"
    ws.auto_filter.ref = ws.dimensions

    from openpyxl.styles import Alignment, Font
    from openpyxl.utils import get_column_letter

    for column_index, name in enumerate(
        review.columns, start=1
    ):
        letter = get_column_letter(column_index)
        ws.column_dimensions[letter].width = (
            75 if name.startswith("text_") else 28
        )

        if name == "id":
            for row in range(2, ws.max_row + 1):
                cell = ws.cell(row, column_index)
                cell.number_format = "@"

    for cell in ws[1]:
        cell.font = Font(bold=True)
        cell.alignment = Alignment(wrap_text=True)

print("=" * 60)
print("04A — SEMANTIC REVIEW EXPORT")
print("=" * 60)
print("All tweets       :", len(social_df))
print("Flagged for review:", len(review))
print("Review Excel     :", REVIEW_PATH)
print("Full audit CSV   :", FULL_AUDIT_PATH)
print("Rows deleted     : 0")
print("Source modified  : NO")
print("=" * 60)


04A — SEMANTIC REVIEW EXPORT
All tweets       : 381
Flagged for review: 5
Review Excel     : C:\Penelitian Sentimen Bencana Nasional\02_Cleaning\outputs\audits\part4_semantic_review.xlsx
Full audit CSV   : C:\Penelitian Sentimen Bencana Nasional\02_Cleaning\outputs\audits\part4_full_audit.csv
Rows deleted     : 0
Source modified  : NO
